# Morphology-aware Old Tupi tokenizer/canonicalizer proof of concept

This notebook is a repeatable research experiment for a tokenizer that learns from the pydicate + oldtupicorpus ecosystem.

The target shape remains:

```text
raw Old Tupi surface text
-> normalization / orthography handling
-> morpheme and allomorph segmentation
-> grammar-aware canonical stream
-> reversible-ish inspection
```

This is not ordinary BPE. BPE would learn frequent string fragments, but it would not know that `r` can be a pluriform prefix, that a surface token has person/role features, or that pydicate-rendered analyses can teach canonical morphology. Here the pydicate corpus is the teacher, and the experiment should improve when the corpus grows.

## 1. Setup

Run from the `oldtupicorpus` repo root. The setup cell finds the root, adds `../nhe-enga/tupi` and `../nhe-enga/pydicate` to `sys.path`, then checks the local imports used by corpus generation.

In [1]:
from __future__ import annotations

import json
import random
import re
import subprocess
import sys
from collections import Counter
from pathlib import Path
from pprint import pprint


def find_repo_root(start: Path | None = None) -> Path:
    here = (start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "tokenizer").is_dir() and (candidate / "historic").is_dir():
            return candidate
    raise RuntimeError("Could not find oldtupicorpus repo root from the current directory")


ROOT = find_repo_root()
NHE_ENGA = (ROOT.parent / "nhe-enga").resolve()
PATHS_TO_ADD = [ROOT, NHE_ENGA / "tupi", NHE_ENGA / "pydicate"]

for path in reversed(PATHS_TO_ADD):
    path_str = str(path)
    if path.exists() and path_str not in sys.path:
        sys.path.insert(0, path_str)

print(f"Repo root: {ROOT}")
for path in PATHS_TO_ADD:
    print(f"sys.path entry {'OK' if path.exists() else 'MISSING'}: {path}")

from tokenizer.morph_poc_utils import (
    FactorizationConfig,
    MorphBaseline,
    append_jsonl,
    build_morph_rows,
    count_by,
    evaluate_prediction_fn,
    format_metrics_table,
    inspect_tokens as inspect_tokens_with_registry,
    iter_jsonl,
    load_json,
    load_jsonl,
    load_registry,
    mismatch_summary,
    normalize_surface,
    token_prf,
    utc_now_iso,
    write_morph_dataset,
)

IMPORT_STATUS = {}

try:
    import tupi
    from tupi import TupiAntigo
    IMPORT_STATUS["tupi"] = True
    print(f"OK: import tupi -> {getattr(tupi, '__file__', '(namespace package)')}")
except Exception as exc:
    IMPORT_STATUS["tupi"] = False
    print(f"FAIL: import tupi: {type(exc).__name__}: {exc}")

try:
    from pydicate.lang.tupilang import *  # noqa: F401,F403
    IMPORT_STATUS["pydicate.lang.tupilang"] = True
    print("OK: from pydicate.lang.tupilang import *")
except Exception as exc:
    IMPORT_STATUS["pydicate.lang.tupilang"] = False
    print(f"FAIL: from pydicate.lang.tupilang import *: {type(exc).__name__}: {exc}")

try:
    from pydicate.lang.tupilang.pos import *  # noqa: F401,F403
    IMPORT_STATUS["pydicate.lang.tupilang.pos"] = True
    print("OK: from pydicate.lang.tupilang.pos import *")
except Exception as exc:
    IMPORT_STATUS["pydicate.lang.tupilang.pos"] = False
    print(f"FAIL: from pydicate.lang.tupilang.pos import *: {type(exc).__name__}: {exc}")

if not all(IMPORT_STATUS.values()):
    print("\nSome imports failed. Existing artifacts can still be inspected, but rebuilding may fail until ../nhe-enga imports work.")

Repo root: /Users/kian/code/oldtupicorpus
sys.path entry OK: /Users/kian/code/oldtupicorpus
sys.path entry OK: /Users/kian/code/nhe-enga/tupi
sys.path entry OK: /Users/kian/code/nhe-enga/pydicate
OK: import tupi -> /Users/kian/code/nhe-enga/tupi/tupi/__init__.py
OK: from pydicate.lang.tupilang import *
OK: from pydicate.lang.tupilang.pos import *


## 2. Experiment configuration

These are the main knobs. Set `FORCE_REBUILD = True` when you have added pydicate-encoded data to `historic/` or `synthetic/` and want the corpus, factorized dataset, model, metrics, and history to reflect it.

In [2]:
# Corpus rebuild controls
FORCE_REBUILD = True
INCLUDE_SYNTHETIC = True
LABEL_FROM_ANNOTATED = True
ORTH_EXPAND = ["POTIGUARA", "TUPINAMBA", "SEM_DIACRITICO"]
ORTH_EXPAND_ALL = False
ORTH_WORKERS = 1
ORTH_BATCH_SIZE = 200
BUILD_LOG_EVERY = 10000

# Factorized target controls
DROP_FEATURE_PREFIXES = {"DEEPEST_NODE"}
DROP_FEATURES = {"DIRECT"}
KEEP_ROOT_FEATURE = True
USE_EXPLICIT_S_IDS = False

# Dataset and split controls
MAX_EXAMPLES = 5000
RANDOM_SEED = 42
MIN_INPUT_LEN = 1
MAX_SRC_LEN = 220
MAX_TGT_LEN = 320
DEV_FRACTION = 0.2
EVAL_LIMIT = 200
QUALITATIVE_EXAMPLES = 10

# Neural model controls
TRAIN_MODEL = True
SAVE_CHECKPOINT = True
CHECKPOINT_PATH = "tokenizer/output/morph_tokenizer_poc.pt"
EPOCHS = 3
BATCH_SIZE = 16
EMBED_DIM = 64
HIDDEN_DIM = 128
LEARNING_RATE = 3e-3
TEACHER_FORCING = 0.85
MAX_DECODE_LEN = 160

# Experiment logging
WRITE_EXPERIMENT_HISTORY = True
HISTORY_PATH = "tokenizer/output/morph_experiment_history.jsonl"

BUILD_CONFIG = {
    "force_rebuild": FORCE_REBUILD,
    "include_synthetic": INCLUDE_SYNTHETIC,
    "label_from_annotated": LABEL_FROM_ANNOTATED,
    "orth_expand": ORTH_EXPAND,
    "orth_expand_all": ORTH_EXPAND_ALL,
    "orth_workers": ORTH_WORKERS,
    "orth_batch_size": ORTH_BATCH_SIZE,
    "build_log_every": BUILD_LOG_EVERY,
}

TRAINING_CONFIG = {
    "max_examples": MAX_EXAMPLES,
    "random_seed": RANDOM_SEED,
    "min_input_len": MIN_INPUT_LEN,
    "max_src_len": MAX_SRC_LEN,
    "max_tgt_len": MAX_TGT_LEN,
    "dev_fraction": DEV_FRACTION,
    "train_model": TRAIN_MODEL,
    "save_checkpoint": SAVE_CHECKPOINT,
    "checkpoint_path": CHECKPOINT_PATH,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "embed_dim": EMBED_DIM,
    "hidden_dim": HIDDEN_DIM,
    "learning_rate": LEARNING_RATE,
    "teacher_forcing": TEACHER_FORCING,
}

FACTORIZATION_CONFIG = FactorizationConfig(
    drop_feature_prefixes=set(DROP_FEATURE_PREFIXES),
    drop_features=set(DROP_FEATURES),
    keep_root_feature=KEEP_ROOT_FEATURE,
    use_explicit_s_ids=USE_EXPLICIT_S_IDS,
)

pprint({"build": BUILD_CONFIG, "factorization": FACTORIZATION_CONFIG.to_json(), "training": TRAINING_CONFIG})

{'build': {'build_log_every': 10000,
           'force_rebuild': True,
           'include_synthetic': True,
           'label_from_annotated': True,
           'orth_batch_size': 200,
           'orth_expand': ['POTIGUARA', 'TUPINAMBA', 'SEM_DIACRITICO'],
           'orth_expand_all': False,
           'orth_workers': 1},
 'factorization': {'drop_feature_prefixes': ['DEEPEST_NODE'],
                   'drop_features': ['DIRECT'],
                   'keep_root_feature': True,
                   'use_explicit_s_ids': False},
 'training': {'batch_size': 16,
              'checkpoint_path': 'tokenizer/output/morph_tokenizer_poc.pt',
              'dev_fraction': 0.2,
              'embed_dim': 64,
              'epochs': 3,
              'hidden_dim': 128,
              'learning_rate': 0.003,
              'max_examples': 5000,
              'max_src_len': 220,
              'max_tgt_len': 320,
              'min_input_len': 1,
              'random_seed': 42,
              'save_checkpo

## 3. Rebuild or load tokenizer artifacts

The notebook still relies on the existing scripts for source discovery and canonical ID generation:

- `tokenizer/build_corpus_json.py`
- `tokenizer/rawgrammarpair.py`

The difference from the first proof of concept is that rebuild behavior is explicit and controlled by the config above.

In [3]:
OUT_DIR = ROOT / "tokenizer" / "output"
CORPUS_JSONL = OUT_DIR / "corpus.jsonl"
CANONICAL_IO = OUT_DIR / "canonical_io.jsonl"
TOKENS_JSON = OUT_DIR / "annotated_tokens.json"
TAGS_JSON = OUT_DIR / "annotated_tags.json"
SUBTAGS_JSON = OUT_DIR / "annotated_subtags.json"
TOKEN_PAIRS_JSON = OUT_DIR / "annotated_token_pairs.json"
VARIANTS_JSON = OUT_DIR / "annotated_token_variants.json"
MORPH_IO = OUT_DIR / "morph_io.jsonl"
MORPH_VOCAB = OUT_DIR / "morph_vocab.json"
MORPH_META = OUT_DIR / "morph_dataset_meta.json"
HISTORY_FILE = ROOT / HISTORY_PATH
CHECKPOINT_FILE = ROOT / CHECKPOINT_PATH


def run_repo_command(args: list[str]) -> None:
    print("$", " ".join(args))
    result = subprocess.run(args, cwd=ROOT, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout[-6000:])
    if result.stderr:
        print(result.stderr[-6000:])
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(args)}")


OUT_DIR.mkdir(parents=True, exist_ok=True)

build_cmd = [
    "python3",
    "tokenizer/build_corpus_json.py",
    "--out_jsonl",
    "tokenizer/output/corpus.jsonl",
]
if INCLUDE_SYNTHETIC:
    build_cmd.append("--include-synthetic")
if LABEL_FROM_ANNOTATED:
    build_cmd.append("--label-from-annotated")
if ORTH_EXPAND:
    build_cmd.extend(["--orth-expand", *ORTH_EXPAND])
if ORTH_EXPAND_ALL:
    build_cmd.append("--orth-expand-all")
if ORTH_WORKERS:
    build_cmd.extend(["--orth-workers", str(ORTH_WORKERS)])
if ORTH_BATCH_SIZE:
    build_cmd.extend(["--orth-batch-size", str(ORTH_BATCH_SIZE)])
if BUILD_LOG_EVERY:
    build_cmd.extend(["--log-every", str(BUILD_LOG_EVERY)])

rawgrammar_cmd = [
    "python3",
    "tokenizer/rawgrammarpair.py",
    "--in_json",
    "tokenizer/output/corpus.jsonl",
    "--out_dir",
    "tokenizer/output",
]
if BUILD_LOG_EVERY:
    rawgrammar_cmd.extend(["--log-every", str(BUILD_LOG_EVERY)])

if FORCE_REBUILD or not CORPUS_JSONL.exists():
    run_repo_command(build_cmd)
else:
    print(f"Loading existing {CORPUS_JSONL.relative_to(ROOT)}")

core_outputs = [CANONICAL_IO, TOKENS_JSON, TAGS_JSON, SUBTAGS_JSON]
if FORCE_REBUILD or not all(path.exists() for path in core_outputs):
    run_repo_command(rawgrammar_cmd)
else:
    print("Loading existing canonical IO and registries")

for path in [CORPUS_JSONL, CANONICAL_IO, TOKENS_JSON, TAGS_JSON, SUBTAGS_JSON, TOKEN_PAIRS_JSON, VARIANTS_JSON]:
    print(f"{path.relative_to(ROOT)}: {'present' if path.exists() else 'missing'}")

$ python3 tokenizer/build_corpus_json.py --out_jsonl tokenizer/output/corpus.jsonl --include-synthetic --label-from-annotated --orth-expand POTIGUARA TUPINAMBA SEM_DIACRITICO --orth-workers 1 --orth-batch-size 200 --log-every 10000
[corpus] rows=20000 skipped=0 rate=6596.4/s
[corpus] rows=60000 skipped=0 rate=6712.4/s
[corpus] rows=140000 skipped=0 rate=6968.0/s
[corpus] rows=210000 skipped=0 rate=7005.0/s
[corpus] rows=260000 skipped=0 rate=7044.5/s
[corpus] rows=270000 skipped=0 rate=7017.5/s
[corpus] rows=280000 skipped=0 rate=7025.4/s
[corpus] rows=360000 skipped=0 rate=6899.8/s
[corpus] rows=390000 skipped=0 rate=6907.6/s
[corpus] rows=410000 skipped=0 rate=6893.3/s
[corpus] rows=420000 skipped=0 rate=6892.9/s
[corpus] rows=440000 skipped=0 rate=6870.7/s
[corpus] rows=500000 skipped=0 rate=7021.8/s
[corpus] rows=520000 skipped=0 rate=6995.1/s
[corpus] rows=600000 skipped=0 rate=6981.8/s
[corpus] rows=650000 skipped=0 rate=6924.0/s
[corpus] rows=670000 skipped=0 rate=6848.2/s
[corp

## 4. Factorize and write the reusable morph dataset

This section converts opaque canonical IDs into the model-facing stream and writes reusable artifacts:

- `tokenizer/output/morph_io.jsonl`
- `tokenizer/output/morph_vocab.json`
- `tokenizer/output/morph_dataset_meta.json`

`ROOT` is kept by default as `<G:ROOT>` because it is linguistically useful. Generated structure such as `DEEPEST_NODE_*` and debug relation `DIRECT` are dropped by default.

In [4]:
corpus_rows = load_jsonl(CORPUS_JSONL)
canonical_rows = load_jsonl(CANONICAL_IO)
token_items, id_to_morpheme, morpheme_to_id = load_registry(TOKENS_JSON, "value")
tag_items, id_to_tag, tag_to_id = load_registry(TAGS_JSON, "tag")
subtag_items, id_to_subtag, subtag_to_id = load_registry(SUBTAGS_JSON, "subtag")
token_pairs = load_json(TOKEN_PAIRS_JSON, default=[])
token_variants = load_json(VARIANTS_JSON, default=[])

morph_rows, morph_build_stats = build_morph_rows(
    corpus_rows,
    canonical_rows,
    id_to_tag,
    id_to_subtag,
    FACTORIZATION_CONFIG,
)
morph_vocab, morph_meta = write_morph_dataset(
    MORPH_IO,
    MORPH_VOCAB,
    MORPH_META,
    morph_rows,
    corpus_rows,
    BUILD_CONFIG,
    FACTORIZATION_CONFIG,
)

BASELINE = MorphBaseline(
    id_to_morpheme=id_to_morpheme,
    id_to_tag=id_to_tag,
    canonical_rows=canonical_rows,
    token_variants=token_variants,
    factorization_config=FACTORIZATION_CONFIG,
)


def baseline_tokenize(text: str) -> list[str]:
    return BASELINE.tokenize(text)


def baseline_tokenize_batch(texts: list[str]) -> list[list[str]]:
    return BASELINE.tokenize_batch(texts)


def baseline_raw_rate(pred_tokens: list[str]) -> float:
    return BASELINE.raw_rate(pred_tokens)


def inspect_tokens(tokens: list[str]) -> list[dict]:
    return inspect_tokens_with_registry(tokens, id_to_morpheme)


sample_for_raw = morph_rows[: min(1000, len(morph_rows))]
baseline_raw_oov_rate = (
    sum(baseline_raw_rate(baseline_tokenize(row["input"])) for row in sample_for_raw) / max(1, len(sample_for_raw))
)
corpus_summary = morph_meta["counts"]

print("Corpus and dataset counts")
print(f"  corpus rows:              {len(corpus_rows)}")
print(f"  canonical rows:           {len(canonical_rows)}")
print(f"  morph rows:               {len(morph_rows)}")
print(f"  historic rows:            {corpus_summary['historic_rows']}")
print(f"  synthetic rows:           {corpus_summary['synthetic_rows']}")
print(f"  orthographic variant rows:{corpus_summary['orthographic_variant_rows']}")
print(f"  M-token count:            {len(morph_vocab['m_tokens'])}")
print(f"  G-feature count:          {len(morph_vocab['g_tokens'])}")
print(f"  baseline RAW/OOV rate:    {baseline_raw_oov_rate:.3%} on {len(sample_for_raw)} rows")
print(f"  stale variant rows ignored by baseline: {BASELINE.stale_variant_rows}")
print("\nCounts by corpus:")
pprint(corpus_summary["by_corpus"])
print("\nCounts by orthography:")
pprint(corpus_summary["by_orth"])
print("\nWrote:")
for path in [MORPH_IO, MORPH_VOCAB, MORPH_META]:
    print(f"  {path.relative_to(ROOT)}")

Corpus and dataset counts
  corpus rows:              2014384
  canonical rows:           2014384
  morph rows:               2014384
  historic rows:            358
  synthetic rows:           2014026
  orthographic variant rows:1319996
  M-token count:            5200
  G-feature count:          101
  baseline RAW/OOV rate:    2.391% on 1000 rows
  stale variant rows ignored by baseline: 0

Counts by corpus:
{'historic': 358, 'synthetic': 2014026}

Counts by orthography:
{'NAVARRO': 694388,
 'POTIGUARA': 539119,
 'SEM_DIACRITICO': 642854,
 'TUPINAMBA': 138023}

Wrote:
  tokenizer/output/morph_io.jsonl
  tokenizer/output/morph_vocab.json
  tokenizer/output/morph_dataset_meta.json


In [5]:
def compact_tokens(tokens: list[str] | str, max_tokens: int = 80) -> str:
    if isinstance(tokens, str):
        tokens = tokens.split()
    if len(tokens) <= max_tokens:
        return " ".join(tokens)
    return " ".join(tokens[:max_tokens]) + f" ... (+{len(tokens) - max_tokens} tokens)"


print("corpus.jsonl examples")
for row in corpus_rows[:3]:
    pprint(row)

print("\nmorph_io.jsonl examples")
for row in morph_rows[:3]:
    pprint(row)

print("\nregistry examples")
print("M:")
pprint(token_items[:5])
print("T:")
pprint(tag_items[:5])
print("S:")
pprint(subtag_items[:8])
if token_variants:
    print("variants:")
    pprint(token_variants[:5])

corpus.jsonl examples
{'anotated': 'Santa '
             "Cruz[DEEPEST_NODE_7:DIRECT:OBJECT:PROPER_NOUN]r[DEEPEST_NODE_5:PLURIFORM_PREFIX:R]a'ang[DEEPEST_NODE_6:ROOT]ab[DEEPEST_NODE_5:FACILITY_SUFFIX]a[CONSONANT_ENDING:DEEPEST_NODE_5:NOUN:POSSESSOR:SUBSTANTIVE_SUFFIX] "
             'r[DEEPEST_NODE_4:PLURIFORM_PREFIX:R]esé[DEEPEST_NODE_4:POSTPOSITION] '
             'oré[1ppe:DEEPEST_NODE_3:OBJECT:PRONOUN]pysyrõ[DEEPEST_NODE_1:ROOT] '
             'îepé[2ps:DEEPEST_NODE_1:OBJECT_1P:PRONOUN:SUBJECT] '
             'Tupã[DEEPEST_NODE_9:PROPER_NOUN] '
             'oré[1ppe:DEEPEST_NODE_11:OBJECT:POSSESSIVE_PRONOUN:PRONOUN] '
             'îar[DEEPEST_NODE_10:ROOT:VOCATIVE] '
             "oré[1ppe:DEEPEST_NODE_15:OBJECT:PRONOUN]amotar[DEEPEST_NODE_14:ROOT]e'ym[DEEPEST_NODE_13:NEGATION_SUFFIX]bar[ABSOLUTE_AGENT_SUFFIX:DEEPEST_NODE_13]a[CONSONANT_ENDING:DEEPEST_NODE_13:SUBSTANTIVE_SUFFIX] "
             'suí[DEEPEST_NODE_12:POSTPOSITION]',
 'corpus': 'historic',
 'index': 0,
 'label': "San

## 5. Baseline tokenizer/canonicalizer

The baseline is a control model. It is intentionally transparent:

- Unicode NFC + whitespace normalization
- longest-match segmentation within each whitespace word
- longer known morphemes are preferred
- `annotated_token_variants.json` contributes variant-to-canonical candidates when the IDs exist in the current registry
- unknown chunks are emitted as `<RAW:...>`
- grammar features are estimated from the most frequent observed feature sequence for the M-token

In [6]:
for text in [
    "amém",
    "tuba ta'yra Espírito Santo rera pupé",
    "orépysyrõte îepé mba'eaíba suí",
    "xe rera",
    "xerera",
]:
    pred = baseline_tokenize(text)
    print("\n---")
    print("INPUT:", text)
    print("BASELINE:", compact_tokens(pred, max_tokens=100))
    print(f"RAW/OOV rate: {baseline_raw_rate(pred):.2%}")
    pprint(inspect_tokens(pred)[:20])


---
INPUT: amém
BASELINE: <M:000024> <G:AMEN> <G:INTERJECTION>
RAW/OOV rate: 0.00%
[{'grammar': ['AMEN', 'INTERJECTION'],
  'raw': False,
  'surface': 'amém',
  'token': '<M:000024>'}]

---
INPUT: tuba ta'yra Espírito Santo rera pupé
BASELINE: <M:004230> <RAW:b> <M:000006> <G:1ps> <G:SUBJECT_PREFIX> <M:000357> <G:CONSONANT> <G:PERMISSIVE_PREFIX> <RAW:'> <M:000026> <G:CLITIC> <G:CONSONANT_ENDING> <G:SUBSTANTIVE_SUFFIX> <M:003931> <M:000020> <M:000021> <M:003933> <M:003931> <M:000023> <G:POSTPOSITION>
RAW/OOV rate: 16.67%
[{'grammar': [], 'raw': False, 'surface': 'tu', 'token': '<M:004230>'},
 {'grammar': [], 'raw': True, 'surface': 'b', 'token': '<RAW:b>'},
 {'grammar': ['1ps', 'SUBJECT_PREFIX'],
  'raw': False,
  'surface': 'a',
  'token': '<M:000006>'},
 {'grammar': ['CONSONANT', 'PERMISSIVE_PREFIX'],
  'raw': False,
  'surface': 'ta',
  'token': '<M:000357>'},
 {'grammar': [], 'raw': True, 'surface': "'", 'token': "<RAW:'>"},
 {'grammar': ['CLITIC', 'CONSONANT_ENDING', 'SUBSTANTIVE_

## 6. Train/dev split from `morph_io.jsonl`

The training source is now the reusable factorized dataset, not notebook-local variables. The split is deterministic with `RANDOM_SEED`; rows that are empty or too long are skipped and counted.

In [7]:
PAD = "<PAD>"
BOS = "<BOS>"
EOS = "<EOS>"
UNK = "<UNK>"

all_morph_rows = load_jsonl(MORPH_IO)
loaded_vocab = load_json(MORPH_VOCAB)

skip_counts = Counter()
eligible_rows = []
for row in all_morph_rows:
    input_text = normalize_surface(str(row.get("input", "")))
    target_tokens = str(row.get("target", "")).split()
    if len(input_text) < MIN_INPUT_LEN:
        skip_counts["too_short_input"] += 1
        continue
    if len(input_text) > MAX_SRC_LEN:
        skip_counts["too_long_input"] += 1
        continue
    if not target_tokens:
        skip_counts["empty_target"] += 1
        continue
    if len(target_tokens) + 2 > MAX_TGT_LEN:
        skip_counts["too_long_target"] += 1
        continue
    next_row = dict(row)
    next_row["input"] = input_text
    next_row["target_tokens"] = target_tokens
    eligible_rows.append(next_row)

rng = random.Random(RANDOM_SEED)
rng.shuffle(eligible_rows)
selected_rows = eligible_rows[:MAX_EXAMPLES]
split_at = max(1, int(len(selected_rows) * (1.0 - DEV_FRACTION))) if len(selected_rows) > 1 else len(selected_rows)
train_examples = selected_rows[:split_at]
dev_examples = selected_rows[split_at:] or selected_rows[: min(10, len(selected_rows))]

src_chars = sorted({ch for row in train_examples for ch in row["input"]})
src_itos = [PAD, BOS, EOS, UNK] + src_chars
src_stoi = {tok: i for i, tok in enumerate(src_itos)}

tgt_itos = loaded_vocab["tokens"]
for required in [PAD, BOS, EOS, UNK]:
    if required not in tgt_itos:
        tgt_itos.insert(0, required)
tgt_stoi = {tok: i for i, tok in enumerate(tgt_itos)}

SRC_PAD_IDX = src_stoi[PAD]
TGT_PAD_IDX = tgt_stoi[PAD]


def encode_source(text: str) -> list[int]:
    text = normalize_surface(text)[:MAX_SRC_LEN]
    ids = [src_stoi[BOS]]
    ids.extend(src_stoi.get(ch, src_stoi[UNK]) for ch in text)
    ids.append(src_stoi[EOS])
    return ids


def encode_target(tokens: list[str]) -> list[int]:
    clipped = tokens[: MAX_TGT_LEN - 2]
    ids = [tgt_stoi[BOS]]
    ids.extend(tgt_stoi.get(tok, tgt_stoi[UNK]) for tok in clipped)
    ids.append(tgt_stoi[EOS])
    return ids


def decode_target(ids: list[int]) -> list[str]:
    out = []
    for idx in ids:
        tok = tgt_itos[int(idx)]
        if tok == EOS:
            break
        if tok not in {PAD, BOS}:
            out.append(tok)
    return out


print(f"morph rows loaded: {len(all_morph_rows)}")
print(f"eligible rows:     {len(eligible_rows)}")
print(f"selected rows:     {len(selected_rows)} / MAX_EXAMPLES={MAX_EXAMPLES}")
print(f"train/dev:         {len(train_examples)} / {len(dev_examples)}")
print(f"skipped rows:      {dict(skip_counts)}")
print(f"source char vocab: {len(src_itos)}")
print(f"target vocab:      {len(tgt_itos)}")
if selected_rows:
    print("\nexample input:", selected_rows[0]["input"])
    print("example target:", compact_tokens(selected_rows[0]["target_tokens"], max_tokens=100))

morph rows loaded: 2014384
eligible rows:     2014384
selected rows:     5000 / MAX_EXAMPLES=5000
train/dev:         4000 / 1000
skipped rows:      {}
source char vocab: 43
target vocab:      5306

example input: taatyguasu ume ixe
example target: <M:000017> <G:PERMISSIVE_PREFIX> <G:VOWEL> <M:000006> <G:1ps> <G:SUBJECT_PREFIX> <M:000823> <M:000251> <G:NEGATION_PARTICLE> <G:UME> <M:000354> <G:1ps> <G:PRONOUN> <G:SUBJECT>


## 7. Small PyTorch encoder-decoder with attention

This remains intentionally small and local. It downloads no pretrained model.

Install dependencies if needed:

```bash
python3 -m pip install torch numpy notebook ipykernel
```

If `TRAIN_MODEL = False` and `CHECKPOINT_PATH` exists, the cell loads the checkpoint for inference.

In [8]:
try:
    import torch
    import torch.nn as nn
    TORCH_AVAILABLE = True
    print(f"torch: {torch.__version__}")
except ModuleNotFoundError:
    TORCH_AVAILABLE = False
    print("Torch is not installed. Install it to run neural training:")
    print("  python3 -m pip install torch numpy notebook ipykernel")

NEURAL_READY = False
neural_model = None
checkpoint_saved_path = None

if TORCH_AVAILABLE and selected_rows:
    def choose_device():
        if torch.cuda.is_available():
            return torch.device("cuda")
        if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            return torch.device("mps")
        return torch.device("cpu")

    device = choose_device()
    print("device:", device)

    def pad_sequences(seqs: list[list[int]], pad_idx: int):
        max_len = max(len(seq) for seq in seqs)
        tensor = torch.full((len(seqs), max_len), pad_idx, dtype=torch.long)
        for i, seq in enumerate(seqs):
            tensor[i, : len(seq)] = torch.tensor(seq, dtype=torch.long)
        return tensor

    def make_batches(examples: list[dict], batch_size: int, shuffle: bool = True):
        rows = examples[:]
        if shuffle:
            random.shuffle(rows)
        for start in range(0, len(rows), batch_size):
            batch = rows[start : start + batch_size]
            src = pad_sequences([encode_source(row["input"]) for row in batch], SRC_PAD_IDX)
            tgt = pad_sequences([encode_target(row["target_tokens"]) for row in batch], TGT_PAD_IDX)
            yield src, tgt, batch

    class Encoder(nn.Module):
        def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int, pad_idx: int):
            super().__init__()
            self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
            self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
            self.hidden_proj = nn.Linear(hidden_dim * 2, hidden_dim)

        def forward(self, src):
            emb = self.embedding(src)
            outputs, hidden = self.gru(emb)
            hidden_cat = torch.cat([hidden[-2], hidden[-1]], dim=1)
            hidden0 = torch.tanh(self.hidden_proj(hidden_cat)).unsqueeze(0)
            return outputs, hidden0

    class AttentionDecoder(nn.Module):
        def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int, enc_dim: int, pad_idx: int):
            super().__init__()
            self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
            self.enc_proj = nn.Linear(enc_dim, hidden_dim)
            self.gru = nn.GRU(embed_dim + enc_dim, hidden_dim, batch_first=True)
            self.out = nn.Linear(hidden_dim + enc_dim + embed_dim, vocab_size)

        def step(self, prev_tok, hidden, enc_outputs, src_mask):
            emb = self.embedding(prev_tok).unsqueeze(1)
            projected = self.enc_proj(enc_outputs)
            scores = torch.bmm(projected, hidden[-1].unsqueeze(2)).squeeze(2)
            scores = scores.masked_fill(~src_mask, -1e9)
            weights = torch.softmax(scores, dim=1).unsqueeze(1)
            context = torch.bmm(weights, enc_outputs)
            output, hidden = self.gru(torch.cat([emb, context], dim=2), hidden)
            logits = self.out(torch.cat([output.squeeze(1), context.squeeze(1), emb.squeeze(1)], dim=1))
            return logits, hidden

    class Seq2Seq(nn.Module):
        def __init__(self, src_vocab: int, tgt_vocab: int, embed_dim: int, hidden_dim: int):
            super().__init__()
            self.encoder = Encoder(src_vocab, embed_dim, hidden_dim, SRC_PAD_IDX)
            self.decoder = AttentionDecoder(tgt_vocab, embed_dim, hidden_dim, hidden_dim * 2, TGT_PAD_IDX)

        def forward(self, src, tgt, teacher_forcing: float = 1.0):
            src_mask = src != SRC_PAD_IDX
            enc_outputs, hidden = self.encoder(src)
            prev_tok = tgt[:, 0]
            logits_by_t = []
            for t in range(1, tgt.size(1)):
                logits, hidden = self.decoder.step(prev_tok, hidden, enc_outputs, src_mask)
                logits_by_t.append(logits.unsqueeze(1))
                predicted = logits.argmax(dim=1)
                prev_tok = tgt[:, t] if random.random() < teacher_forcing else predicted
            return torch.cat(logits_by_t, dim=1)

    def rebuild_vocabs_from_checkpoint(checkpoint: dict) -> None:
        global src_itos, src_stoi, tgt_itos, tgt_stoi, SRC_PAD_IDX, TGT_PAD_IDX
        src_itos = checkpoint["src_itos"]
        tgt_itos = checkpoint["tgt_itos"]
        src_stoi = {tok: i for i, tok in enumerate(src_itos)}
        tgt_stoi = {tok: i for i, tok in enumerate(tgt_itos)}
        SRC_PAD_IDX = src_stoi[PAD]
        TGT_PAD_IDX = tgt_stoi[PAD]

    checkpoint_exists = CHECKPOINT_FILE.exists()
    model_embed_dim = EMBED_DIM
    model_hidden_dim = HIDDEN_DIM

    if not TRAIN_MODEL and checkpoint_exists:
        checkpoint = torch.load(CHECKPOINT_FILE, map_location=device)
        rebuild_vocabs_from_checkpoint(checkpoint)
        model_config = checkpoint.get("model_config", {})
        model_embed_dim = int(model_config.get("embed_dim", EMBED_DIM))
        model_hidden_dim = int(model_config.get("hidden_dim", HIDDEN_DIM))
        neural_model = Seq2Seq(len(src_itos), len(tgt_itos), model_embed_dim, model_hidden_dim).to(device)
        neural_model.load_state_dict(checkpoint["model_state"])
        NEURAL_READY = True
        checkpoint_saved_path = str(CHECKPOINT_FILE.relative_to(ROOT))
        print(f"Loaded checkpoint: {checkpoint_saved_path}")

    elif TRAIN_MODEL:
        torch.manual_seed(RANDOM_SEED)
        random.seed(RANDOM_SEED)
        neural_model = Seq2Seq(len(src_itos), len(tgt_itos), EMBED_DIM, HIDDEN_DIM).to(device)
        optimizer = torch.optim.Adam(neural_model.parameters(), lr=LEARNING_RATE)
        criterion = nn.CrossEntropyLoss(ignore_index=TGT_PAD_IDX)

        for epoch in range(1, EPOCHS + 1):
            neural_model.train()
            losses = []
            for src, tgt, _batch in make_batches(train_examples, BATCH_SIZE, shuffle=True):
                src = src.to(device)
                tgt = tgt.to(device)
                optimizer.zero_grad()
                logits = neural_model(src, tgt, teacher_forcing=TEACHER_FORCING)
                loss = criterion(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1))
                loss.backward()
                torch.nn.utils.clip_grad_norm_(neural_model.parameters(), 1.0)
                optimizer.step()
                losses.append(float(loss.detach().cpu()))
            print(f"epoch {epoch:02d} train_loss={sum(losses) / max(1, len(losses)):.4f}")

        NEURAL_READY = True
        if SAVE_CHECKPOINT:
            CHECKPOINT_FILE.parent.mkdir(parents=True, exist_ok=True)
            torch.save(
                {
                    "model_state": neural_model.state_dict(),
                    "src_itos": src_itos,
                    "tgt_itos": tgt_itos,
                    "model_config": {"embed_dim": EMBED_DIM, "hidden_dim": HIDDEN_DIM},
                    "training_config": TRAINING_CONFIG,
                    "build_config": BUILD_CONFIG,
                    "factorization_config": FACTORIZATION_CONFIG.to_json(),
                },
                CHECKPOINT_FILE,
            )
            checkpoint_saved_path = str(CHECKPOINT_FILE.relative_to(ROOT))
            print(f"Saved checkpoint: {checkpoint_saved_path}")
    else:
        print("TRAIN_MODEL is False and no checkpoint exists; neural inference is unavailable.")

    if NEURAL_READY:
        @torch.no_grad()
        def neural_predict(text: str, max_len: int | None = None) -> list[str]:
            neural_model.eval()
            if max_len is None:
                max_len = min(MAX_TGT_LEN, MAX_DECODE_LEN)
            src_ids = encode_source(text)
            src = torch.tensor([src_ids], dtype=torch.long, device=device)
            src_mask = src != SRC_PAD_IDX
            enc_outputs, hidden = neural_model.encoder(src)
            prev_tok = torch.tensor([tgt_stoi[BOS]], dtype=torch.long, device=device)
            out_ids = []
            repeated_suffix_counts = Counter()
            for _ in range(max_len):
                logits, hidden = neural_model.decoder.step(prev_tok, hidden, enc_outputs, src_mask)
                next_id = int(logits.argmax(dim=1).item())
                if tgt_itos[next_id] == EOS:
                    break
                out_ids.append(next_id)
                if len(out_ids) >= 18:
                    suffix = tuple(out_ids[-6:])
                    repeated_suffix_counts[suffix] += 1
                    if repeated_suffix_counts[suffix] >= 3:
                        out_ids = out_ids[:-6]
                        break
                prev_tok = torch.tensor([next_id], dtype=torch.long, device=device)
            return decode_target(out_ids)

        print("Neural model is ready for evaluation/inference.")
elif TORCH_AVAILABLE:
    print("No selected rows are available after filtering; adjust length limits or rebuild data.")
else:
    def neural_predict(text: str, max_len: int | None = None) -> list[str]:
        raise RuntimeError("Torch is not installed; neural prediction is unavailable")

torch: 2.11.0
device: mps
epoch 01 train_loss=1.7131
epoch 02 train_loss=0.6894
epoch 03 train_loss=0.4951
Saved checkpoint: tokenizer/output/morph_tokenizer_poc.pt
Neural model is ready for evaluation/inference.


## 8. Evaluation and experiment history

Metrics are intentionally compact:

- `exact`: full target sequence match
- `token_acc`: position-wise token accuracy
- `token_f1`: multiset token F1 over the whole target
- `m_f1`: multiset F1 over `<M:...>` tokens only
- `g_f1`: multiset F1 over `<G:...>` tokens only
- `raw_rate`: share of baseline-like morpheme chunks emitted as `<RAW:...>`

Each evaluation appends one row to `tokenizer/output/morph_experiment_history.jsonl` when `WRITE_EXPERIMENT_HISTORY = True`.

In [9]:
eval_examples = dev_examples[:EVAL_LIMIT]
metrics = []

baseline_metrics = evaluate_prediction_fn("baseline", baseline_tokenize, eval_examples)
metrics.append(baseline_metrics)

if NEURAL_READY:
    neural_metrics = evaluate_prediction_fn("neural", neural_predict, eval_examples)
    metrics.append(neural_metrics)
else:
    print("Neural evaluation skipped; train or load a checkpoint first.")

print(format_metrics_table(metrics))

history_row = {
    "timestamp": utc_now_iso(),
    "corpus_row_count": len(corpus_rows),
    "canonical_row_count": len(canonical_rows),
    "morph_row_count": len(morph_rows),
    "train_row_count": len(train_examples),
    "dev_row_count": len(dev_examples),
    "vocab_sizes": {
        "source_chars": len(src_itos),
        "target_tokens": len(tgt_itos),
        "m_tokens": len(morph_vocab["m_tokens"]),
        "g_tokens": len(morph_vocab["g_tokens"]),
    },
    "build_config": BUILD_CONFIG,
    "factorization_config": FACTORIZATION_CONFIG.to_json(),
    "training_config": TRAINING_CONFIG,
    "skip_counts": dict(skip_counts),
    "metrics": metrics,
    "checkpoint_path": checkpoint_saved_path,
}

if WRITE_EXPERIMENT_HISTORY:
    append_jsonl(HISTORY_FILE, history_row)
    print(f"Appended experiment history: {HISTORY_FILE.relative_to(ROOT)}")

model     exact  token_acc  token_f1  m_f1   g_f1   raw_rate
baseline  0.105  0.518      0.703     0.630  0.740  0.014   
neural    0.010  0.779      0.863     0.745  0.931  0.000   
Appended experiment history: tokenizer/output/morph_experiment_history.jsonl


In [10]:
print("Qualitative examples")
for row in eval_examples[:QUALITATIVE_EXAMPLES]:
    gold = row["target_tokens"]
    baseline_pred = baseline_tokenize(row["input"])
    neural_pred = neural_predict(row["input"]) if NEURAL_READY else []
    print("\n" + "=" * 80)
    print("INPUT:   ", row["input"])
    print("GOLD:    ", compact_tokens(gold, max_tokens=100))
    print("BASELINE:", compact_tokens(baseline_pred, max_tokens=100))
    print("  mismatch:", mismatch_summary(baseline_pred, gold))
    if NEURAL_READY:
        print("NEURAL:  ", compact_tokens(neural_pred, max_tokens=100))
        print("  mismatch:", mismatch_summary(neural_pred, gold))
    else:
        print("NEURAL:   unavailable")

Qualitative examples

INPUT:    norepyki peiepe
GOLD:     <M:000350> <G:NEGATION_PREFIX> <M:000224> <G:1ppe> <G:OBJECT> <G:PRONOUN> <M:003821> <M:000030> <G:CONSONANT_ENDING> <G:NEGATION_SUFFIX> <M:000366> <G:2pp> <G:OBJECT_1P> <G:PRONOUN> <G:SUBJECT>
BASELINE: <M:000350> <G:NEGATION_PREFIX> <M:000008> <G:1ppe> <G:OBJECT> <G:PRONOUN> <M:003821> <M:000030> <G:CONSONANT_ENDING> <G:NEGATION_SUFFIX> <M:000364> <G:2pp> <G:OBJECT_1P> <G:PRONOUN> <G:SUBJECT>
  mismatch: @2: pred=<M:000008> gold=<M:000224>; @10: pred=<M:000364> gold=<M:000366>
NEURAL:   <M:000350> <G:NEGATION_PREFIX> <M:000224> <G:1ppe> <G:OBJECT> <G:PRONOUN> <M:002196> <M:000030> <G:CONSONANT_ENDING> <G:NEGATION_SUFFIX> <M:000366> <G:2pp> <G:OBJECT_1P> <G:PRONOUN> <G:SUBJECT>
  mismatch: @6: pred=<M:002196> gold=<M:003821>

INPUT:    orérapearõ
GOLD:     <M:000008> <G:1ppe> <G:OBJECT> <G:PRONOUN> <M:000003> <G:PLURIFORM_PREFIX> <G:R> <M:000674>
BASELINE: <M:000008> <G:1ppe> <G:OBJECT> <G:PRONOUN> <M:003931> <M:003550> <RAW:õ>

## 9. Manual test strings

Edit `TEST_STRINGS` and rerun this cell. It shows baseline output, neural output when available, inspection groups, and RAW/OOV chunks.

In [11]:
TEST_STRINGS = [
    "amém",
    "tuba ta'yra Espírito Santo rera pupé",
    "orépysyrõte îepé mba'eaíba suí",
    "aîpotar nde kûara",
    "xe rera",
    "xerera",
    ""
]

for text in TEST_STRINGS:
    print("\n" + "=" * 80)
    print("INPUT:", text)

    baseline_pred = baseline_tokenize(text)
    baseline_raw_chunks = [tok for tok in baseline_pred if tok.startswith("<RAW:")]
    print("\nBASELINE:", compact_tokens(baseline_pred, max_tokens=140))
    print(f"baseline RAW/OOV rate: {baseline_raw_rate(baseline_pred):.2%}")
    print("RAW/OOV chunks:", baseline_raw_chunks or "none")
    print("inspection:")
    pprint(inspect_tokens(baseline_pred)[:40])

    if NEURAL_READY:
        neural_pred = neural_predict(text)
        neural_raw_chunks = [tok for tok in neural_pred if tok.startswith("<RAW:")]
        print("\nNEURAL:", compact_tokens(neural_pred, max_tokens=140))
        print("RAW/OOV chunks:", neural_raw_chunks or "none")
        print("inspection:")
        pprint(inspect_tokens(neural_pred)[:40])
    else:
        print("\nNEURAL: unavailable; install torch and run the training/checkpoint cell.")


INPUT: amém

BASELINE: <M:000024> <G:AMEN> <G:INTERJECTION>
baseline RAW/OOV rate: 0.00%
RAW/OOV chunks: none
inspection:
[{'grammar': ['AMEN', 'INTERJECTION'],
  'raw': False,
  'surface': 'amém',
  'token': '<M:000024>'}]

NEURAL: <M:000006> <G:1ps> <G:SUBJECT_PREFIX> <M:000030> <G:3p> <G:DEFAULT> <G:OBJECT_MARKER> <M:000592>
RAW/OOV chunks: none
inspection:
[{'grammar': ['1ps', 'SUBJECT_PREFIX'],
  'raw': False,
  'surface': 'a',
  'token': '<M:000006>'},
 {'grammar': ['3p', 'DEFAULT', 'OBJECT_MARKER'],
  'raw': False,
  'surface': 'i',
  'token': '<M:000030>'},
 {'grammar': [], 'raw': False, 'surface': 'amĩ', 'token': '<M:000592>'}]

INPUT: tuba ta'yra Espírito Santo rera pupé

BASELINE: <M:004230> <RAW:b> <M:000006> <G:1ps> <G:SUBJECT_PREFIX> <M:000357> <G:CONSONANT> <G:PERMISSIVE_PREFIX> <RAW:'> <M:000026> <G:CLITIC> <G:CONSONANT_ENDING> <G:SUBSTANTIVE_SUFFIX> <M:003931> <M:000020> <M:000021> <M:003933> <M:003931> <M:000023> <G:POSTPOSITION>
baseline RAW/OOV rate: 16.67%
RAW/OOV

## Current limitations

This is still a proof of concept:

- the baseline is greedy longest-match, not a weighted finite-state segmenter
- spacing is only inspection-level reversible
- feature ordering is inherited from pydicate tag strings and may need a more explicit feature ontology
- the neural model is deliberately small and trained from scratch
- exact match is harsh because some forms have ambiguous analyses

The repeatable part is the important step: when you add pydicate-encoded Old Tupi data, set `FORCE_REBUILD = True`, rerun the notebook, and compare `morph_experiment_history.jsonl`. More rows should improve the factorized dataset, baseline coverage, target vocabulary, and eventually neural generalization.